# 관광지 유형 추천 모델 정리

> 파일명: `target_tourism_type.ipynb`  
> 목적: 관광지를 대상으로 유형을 예측하고, 각 관광지가 유형별로 어느 정도의 성격을 가지는지 확률 컬럼으로 저장한다.

---

## 1. 모델 개요

본 모델은 남양주시 관광지 데이터를 기반으로 관광지의 유형을 예측하는 **관광지 유형 분류 모델**이다.  
관광지의 기본 정보, 콘텐츠 유형, 관광 분류 코드, 위치 정보, 데이터 품질 점수 등을 입력 변수로 사용하여 각 관광지가 어떤 관광 유형에 가까운지 분류한다.

단순히 하나의 유형만 예측하는 것이 아니라, `predict_proba()`를 활용하여 관광지별 **유형 확률값**을 생성하는 것이 핵심이다.  
이 확률값은 이후 숙소 기반 관광지 추천, 테마 기반 추천, 코스 추천 모델의 입력값으로 활용할 수 있다.

---

## 2. 데이터 수집

| 구분 | 내용 |
|---|---|
| 사용 데이터 | `남양주시 관광지.csv` |
| 데이터 출처 | 한국관광공사 국문 관광정보 서비스 |
| API | `areaBasedList2` |
| 대상 지역 | 남양주시 |
| 주요 컬럼 | 관광지명, 주소, 좌표, 콘텐츠 유형, 관광 분류 코드, 이미지, 전화번호 등 |

---

## 3. 데이터 전처리

### 3.1 텍스트 컬럼 전처리

텍스트 컬럼에는 관광지명, 주소, 이미지 URL, 전화번호 등 문자열 데이터가 포함된다.

| 처리 항목 | 처리 내용 |
|---|---|
| 결측치 처리 | 결측값을 빈 문자열 `""`로 대체 |
| 데이터형 변환 | 모든 값을 문자열 타입으로 변환 |
| 공백 제거 | 앞뒤 불필요한 공백 제거 |
| 관광지명 정규화 | 관광지명을 소문자로 변환하여 키워드 매칭에 활용 |

---

### 3.2 수치형 컬럼 전처리

수치형 컬럼에는 콘텐츠 유형 ID, 좌표, 지도 확대 수준, 우편번호, 날짜 기반 변수, 데이터 품질 점수 등이 포함된다.

| 처리 항목 | 처리 내용 |
|---|---|
| 데이터형 변환 | 실수 또는 정수형으로 변환 |
| 변환 실패 처리 | 변환이 불가능한 값은 `NaN`으로 처리 |
| 결측치 처리 | 모델 학습 단계에서 중앙값으로 대체 |
| 스케일링 | `StandardScaler`를 사용하여 표준화 |

---

### 3.3 카테고리 매핑

숫자 코드 형태로 제공되는 `contenttypeid`를 사람이 해석할 수 있는 콘텐츠 유형명으로 변환하였다.

| contenttypeid | 콘텐츠 유형 |
|---:|---|
| 12 | 관광지 |
| 14 | 문화시설 |
| 15 | 축제공연행사 |
| 25 | 여행코스 |
| 28 | 레포츠 |
| 32 | 숙박 |
| 38 | 쇼핑 |
| 39 | 음식 |

매핑되지 않은 값이나 결측치는 `기타`로 처리하였다.

---

### 3.4 날짜 파생 변수 생성

timestamp 형식의 날짜 데이터를 `datetime`으로 변환한 뒤, 다음과 같은 파생 변수를 생성하였다.

| 파생 변수 | 설명 |
|---|---|
| `year` | 수정 또는 생성 연도 |
| `month` | 수정 또는 생성 월 |

이를 통해 관광지 정보가 어느 시점에 관리되었는지 모델 입력에 반영할 수 있도록 하였다.

---

## 4. 타겟 변수 생성

### 4.1 타겟 변수

| 구분 | 변수명 |
|---|---|
| 타겟 변수 | `target_tourism_type` |

`target_tourism_type`은 관광지의 최종 유형을 의미한다.  
해당 변수는 콘텐츠 유형 ID, 관광지명 키워드, 관광 분류 코드를 조합하여 생성하였다.

---

### 4.2 타겟 생성 우선순위

타겟 유형은 다음 순서에 따라 생성하였다.

```text
1순위: contenttypeid 기반 명확한 유형 매핑
2순위: 관광지명 키워드 기반 유형 보정
3순위: 대·중·소분류 코드 기반 유형 보정
4순위: 기타 또는 관광지로 처리
```

---

### 4.3 우선순위 기반 매핑

음식, 쇼핑, 숙박 등 명확한 유형은 관광지명이나 분류 코드보다 `contenttypeid`를 우선 적용하여 강제 분류하였다.

| contenttypeid | 반환 값 | 비고 |
|---:|---|---|
| 39 | 음식 | 음식점 |
| 38 | 쇼핑 | 쇼핑몰, 시장 등 |
| 32 | 숙박 | 호텔, 여관 등 |
| 28 | 레저스포츠 | 테마파크, 스포츠 시설 등 |
| 14 | 문화관광 | 박물관, 미술관, 공연장 등 |
| 15 | 축제행사 | 이벤트 장소 |
| 25 | 여행코스 | 코스 정보 |

---

### 4.4 제목 기반 유형 보정

`contenttypeid`가 12인 일반 관광지이거나, 우선순위 기반 매핑에서 분류되지 않은 경우에는 관광지명에 포함된 키워드를 이용하여 세부 유형을 보정하였다.

| 키워드 예시 | 보정 유형 |
|---|---|
| 산, 계곡, 수목원, 공원 | 자연관광 |
| 유적, 묘, 사찰, 성지 | 역사관광 |
| 박물관, 미술관, 전시관 | 문화관광 |
| 체험, 농장, 마을 | 체험관광 |
| 캠핑, 레포츠, 승마 | 레저스포츠 |

---

### 4.5 분류 코드 기반 보정

제목 기반 키워드로도 유형을 판단하기 어려운 경우에는 한국관광공사의 관광 분류 코드인 `cat1~cat3`, `lclsSystm1~lclsSystm3`를 활용하여 유형을 보정하였다.

---

## 5. 데이터 품질 평가

각 관광지 데이터의 정보 완성도를 평가하기 위해 `info_score`를 생성하였다.  
이는 관광지 정보가 얼마나 잘 관리되고 있는지를 나타내는 지표이며, 추천 모델에서 신뢰도 가중치로 활용할 수 있다.

### 5.1 품질 점수 기준

| 항목 | 점수 |
|---|---:|
| 관광지명 존재 | 10 |
| 기본 주소 존재 | 15 |
| 상세 주소 존재 | 5 |
| 좌표 존재 | 15 |
| 콘텐츠 유형 존재 | 10 |
| 대·중·소분류 존재 | 15 |
| 대표 이미지 존재 | 10 |
| 보조 이미지 존재 | 5 |
| 전화번호 존재 | 5 |
| 우편번호 존재 | 5 |
| 수정일 존재 | 5 |
| **총점** | **100** |

---

### 5.2 품질 등급

`info_score`를 기준으로 데이터 품질을 4단계로 분류하였다.

| 품질 등급 | 의미 |
|---|---|
| 낮음 | 정보가 부족하여 신뢰도가 낮음 |
| 보통 | 기본 정보는 있으나 일부 정보가 부족함 |
| 높음 | 주요 정보가 대부분 존재함 |
| 매우 높음 | 관광지 정보가 충분히 관리되고 있음 |

### 활용 예시

| 활용 목적 | 중요하게 볼 항목 |
|---|---|
| 웹 화면 표시 중심 추천 | 대표 이미지, 보조 이미지 |
| 지도 기반 추천 | 좌표 존재 여부 |
| 예약·문의 연계 추천 | 전화번호, 주소 |
| 최신 정보 중심 추천 | 수정일 |

---

## 6. 피처 구성

### 6.1 텍스트 피처

| 피처 | 설명 |
|---|---|
| `title` | 관광지명 |
| `addr1` | 기본 주소 |

텍스트 데이터를 원-핫 인코딩할 경우 관광지명 고유값이 많아져 과적합이 발생할 수 있다.  
따라서 최종 학습 단계에서는 텍스트 피처를 제거하거나 제한적으로 사용하였다.

---

### 6.2 범주형 피처

| 피처 | 설명 |
|---|---|
| `content_type_name` | 콘텐츠 유형명 |
| `cpyrhtDivCd` | 저작권 구분 코드 |
| `cat1` | 대분류 코드 |
| `cat2` | 중분류 코드 |
| `cat3` | 소분류 코드 |
| `lclsSystm1` | 관광 분류 대분류 |
| `lclsSystm2` | 관광 분류 중분류 |
| `lclsSystm3` | 관광 분류 소분류 |
| `info_level` | 데이터 품질 등급 |

범주형 피처는 결측값을 `missing`으로 대체한 뒤 원-핫 인코딩을 적용하였다.

---

### 6.3 수치형 피처

| 피처 | 설명 |
|---|---|
| `contenttypeid` | 콘텐츠 유형 ID |
| `mapx` | 경도 |
| `mapy` | 위도 |
| `mlevel` | 지도 확대 수준 |
| `zipcode` | 우편번호 |
| `year` | 수정 연도 |
| `month` | 수정 월 |
| `info_score` | 데이터 품질 점수 |

수치형 피처는 결측값을 중앙값으로 대체한 뒤 `StandardScaler`를 적용하였다.

---

## 7. 모델 입력 구조

| 구분 | 내용 |
|---|---|
| 입력 변수 `X` | 범주형 피처 + 수치형 피처 |
| 타겟 변수 `y` | `target_tourism_type` |
| 데이터 분할 | Train : Test = 8 : 2 |
| 전처리 방식 | 범주형 원-핫 인코딩, 수치형 중앙값 대체 및 표준화 |

클래스별 데이터 수가 충분한 경우에는 `stratify=y`를 적용하여 학습 데이터와 테스트 데이터의 클래스 비율을 유사하게 유지하였다.

---

## 8. 모델 비교

본 모델에서는 다음 세 가지 분류 모델을 비교하였다.

| 모델 | 특징 |
|---|---|
| RandomForest | 해석이 비교적 쉽고 혼합형 데이터에 안정적 |
| CatBoost | 범주형 데이터 처리에 강점이 있는 부스팅 모델 |
| XGBoost | 높은 예측 성능을 보이는 그래디언트 부스팅 모델 |

---

### 8.1 RandomForest 설정

```python
RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    min_samples_leaf=2,
    n_jobs=-1
)
```

| 파라미터 | 의미 |
|---|---|
| `n_estimators=300` | 300개의 결정트리 생성 |
| `class_weight='balanced'` | 클래스 불균형 보정 |
| `min_samples_leaf=2` | 리프 노드의 최소 샘플 수 제한 |
| `random_state=42` | 실험 재현성 확보 |
| `n_jobs=-1` | 사용 가능한 CPU 코어 전체 사용 |

---

### 8.2 CatBoost 설정

```python
CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_seed=42,
    verbose=0
)
```

CatBoost는 범주형 데이터 처리에 강점이 있으나, 본 비교에서는 동일한 전처리 조건을 맞추기 위해 원-핫 인코딩된 데이터를 입력으로 사용하였다.

---

### 8.3 XGBoost 설정

```python
XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
```

XGBoost는 다중 클래스 확률 예측을 위해 `multi:softprob` 목적 함수를 사용하였다.  
문자열 타겟은 직접 처리하기 어렵기 때문에 `LabelEncoder`를 통해 숫자 라벨로 변환하여 학습한다.

세 모델은 모두 동일한 입력 피처, 동일한 train/test 분할, 동일한 평가 지표를 기준으로 비교하였다.  
이를 통해 특정 모델이 데이터 분할이나 전처리 조건의 차이로 유리해지는 것을 방지하고, 관광지 유형 분류 문제에 가장 적합한 모델을 선정하고자 하였다.

---

## 9. 모델 평가 결과

본 연구에서는 관광지 유형 분류 모델의 성능을 비교하기 위해 RandomForest, XGBoost, CatBoost 세 가지 모델을 동일한 학습 데이터와 테스트 데이터 조건에서 평가하였다.  
평가 지표로는 전체 예측 정확도를 나타내는 Accuracy, 클래스별 성능을 균등하게 반영하는 Macro F1, 데이터 분포를 고려한 Weighted F1을 사용하였다.

### 9.1 모델별 성능 비교

| 순위 | 모델 | Accuracy | Macro F1 | Weighted F1 |
|---:|---|---:|---:|---:|
| 1 | RandomForest | 0.9481 | 0.8413 | 0.9487 |
| 2 | XGBoost | 0.9481 | 0.8290 | 0.9459 |
| 3 | CatBoost | 0.9221 | 0.7848 | 0.9231 |

모델 비교 결과, RandomForest와 XGBoost는 Accuracy 기준으로 동일하게 0.9481의 성능을 보였다.  
그러나 Macro F1과 Weighted F1에서는 RandomForest가 XGBoost보다 더 높은 값을 기록하였다.

관광지 유형 분류 문제는 클래스별 데이터 수가 불균형하므로 Accuracy만으로 모델 성능을 판단하기 어렵다.  
따라서 본 모델에서는 전체 정확도와 클래스별 균형 성능을 함께 고려하여 Macro F1이 가장 높은 RandomForest를 최종 모델로 선정하였다.

### 9.2 최종 선택 모델 성능

| 지표 | 값 |
|---|---:|
| Accuracy | 0.9481 |
| Macro F1 | 0.8413 |
| Weighted F1 | 0.9487 |

최종 선택된 RandomForest 모델은 전체 정확도 94.81%를 기록하였다.  
Weighted F1 역시 0.9487로 높게 나타나 전체 데이터 분포 기준에서는 안정적인 분류 성능을 보였다.

다만 Macro F1은 0.8413으로 Accuracy보다 낮게 나타났다.  
이는 일부 소수 클래스의 예측 성능이 상대적으로 낮기 때문이며, 관광지 유형별 데이터 수 불균형의 영향을 받은 것으로 판단된다.

---

## 10. 클래스별 분류 성능

| 유형 | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| 관광지 | 0.50 | 0.67 | 0.57 | 3 |
| 레저스포츠 | 1.00 | 1.00 | 1.00 | 9 |
| 문화관광 | 1.00 | 1.00 | 1.00 | 3 |
| 쇼핑 | 1.00 | 1.00 | 1.00 | 17 |
| 숙박 | 1.00 | 1.00 | 1.00 | 2 |
| 역사관광 | 1.00 | 0.50 | 0.67 | 2 |
| 음식 | 1.00 | 1.00 | 1.00 | 35 |
| 자연관광 | 0.75 | 0.60 | 0.67 | 5 |
| 체험관광 | 0.50 | 1.00 | 0.67 | 1 |

위 결과는 최종 선택 모델인 RandomForest의 클래스별 분류 성능이다.  
전체 성능은 높지만, 클래스별 support 수에 차이가 있어 소수 클래스의 성능 해석에는 주의가 필요하다.

---

## 11. 평가 해석

모델은 전체적으로 높은 정확도를 보였으며, 특히 다음 유형에서 매우 우수한 분류 성능을 보였다.

- 음식
- 쇼핑
- 레저스포츠
- 문화관광
- 숙박

이 유형들은 `contenttypeid`를 통해 비교적 명확하게 구분된다.  
예를 들어 음식은 `contenttypeid=39`, 쇼핑은 `contenttypeid=38`, 숙박은 `contenttypeid=32`로 구분되므로 모델이 안정적으로 예측할 수 있다.

반면 다음 유형은 상대적으로 성능이 낮게 나타났다.

- 관광지
- 자연관광
- 역사관광
- 체험관광

이 유형들은 대부분 `contenttypeid=12`인 일반 관광지 범주 안에 포함되는 경우가 많다.  
따라서 관광지명 키워드나 분류 코드에 따라 세부 유형을 구분해야 하므로 예측 난이도가 높다.

또한 일부 클래스는 테스트 데이터 수가 매우 적다.  
예를 들어 체험관광은 테스트 데이터에서 1개, 숙박과 역사관광은 각각 2개만 존재한다.  
따라서 해당 클래스의 성능 수치는 데이터 1~2개의 예측 결과에 크게 영향을 받을 수 있다.

---

## 12. 모델 결과물

학습된 모델은 각 관광지에 대해 최종 유형뿐 아니라 유형별 확률값을 생성한다.

| 컬럼명 예시 | 의미 |
|---|---|
| `type_prob_자연관광` | 자연관광일 확률 |
| `type_prob_역사관광` | 역사관광일 확률 |
| `type_prob_문화관광` | 문화관광일 확률 |
| `type_prob_체험관광` | 체험관광일 확률 |
| `type_prob_음식` | 음식일 확률 |
| `type_top1` | 가장 높은 확률의 관광 유형 |
| `type_top1_prob` | 가장 높은 유형 확률 |
| `type_top2` | 두 번째로 높은 확률의 관광 유형 |
| `type_top2_prob` | 두 번째 유형 확률 |

---

## 13. 추천 시스템 활용 방안

관광지 유형 확률값은 이후 추천 시스템에서 다음과 같이 활용할 수 있다.

| 활용 방식 | 설명 |
|---|---|
| 테마 기반 추천 | 사용자가 선택한 테마와 관련 확률이 높은 관광지 추천 |
| 복합 유형 추천 | 자연+역사, 문화+체험 등 복합 성격을 반영 |
| 추천 점수 계산 | 유형 확률을 최종 추천 점수의 가중치로 사용 |
| 코스 다양성 확보 | 같은 유형만 반복되지 않도록 유형 분포 조정 |
| 개인화 추천 | 사용자 선호 테마와 관광지 유형 확률을 매칭 |

---

## 14. 한계점

| 한계 | 설명 |
|---|---|
| 규칙 기반 타겟 생성 | `target_tourism_type`이 사람이 정의한 규칙으로 생성되어 모델도 해당 규칙을 학습하는 구조임 |
| 소수 클래스 부족 | 체험관광, 숙박, 역사관광 등 일부 유형의 데이터 수가 적음 |
| 텍스트 활용 제한 | 관광지명과 주소 텍스트를 제거하여 의미 정보가 충분히 반영되지 못함 |
| 지역 범위 제한 | 남양주시 데이터만 사용하여 다른 지역 적용 시 일반화 성능이 낮을 수 있음 |
| 콘텐츠 ID 의존성 | 음식, 쇼핑, 숙박 등은 콘텐츠 ID만으로 쉽게 구분되어 성능이 과대평가될 수 있음 |

---

## 15. 개선 방향

| 개선 방향 | 설명 |
|---|---|
| 데이터 확장 | 남양주시 외 경기북부 전체 관광지 데이터로 확장 |
| 소수 클래스 보강 | 체험관광, 역사관광, 자연관광 데이터 추가 확보 |
| 텍스트 임베딩 적용 | 관광지명, 개요, 소개글을 TF-IDF 또는 Sentence-BERT로 변환 |
| CatBoost 고유 방식 활용 | 원-핫 인코딩 대신 CatBoost의 범주형 처리 기능 직접 사용 |
| 교차검증 적용 | 단일 train/test split 대신 Stratified K-Fold 적용 |
| 규칙 기반 + ML 혼합 구조 | 명확한 유형은 규칙 기반, 애매한 유형은 ML 기반으로 분리 |
| 추천 모델 연계 | 유형 확률을 관광지 추천 점수 계산에 반영 |

---

## 16. 최종 정리

본 노트북은 남양주시 관광지 데이터를 기반으로 관광지 유형을 예측하는 분류 모델을 구축하였다.  
데이터 전처리 과정에서는 텍스트, 범주형, 수치형 데이터를 정리하고, 콘텐츠 유형 ID와 관광 분류 코드, 관광지명 키워드를 조합하여 `target_tourism_type`을 생성하였다.

모델 학습에는 RandomForest, XGBoost, CatBoost를 비교 대상으로 설정하였다.  

| Accuracy | Macro F1 | Weighted F1 |
|---:|---:|---:|
| 0.9481 | 0.8413 | 0.9487 |

모델 비교 결과 RandomForest와 XGBoost는 Accuracy 기준으로 동일한 0.9481을 기록하였다. 그러나 Macro F1은 RandomForest가 0.8413, XGBoost가 0.8290으로 RandomForest가 더 높게 나타났다. 관광지 유형 분류 문제는 클래스별 데이터 수가 불균형하기 때문에 단순 정확도보다 Macro F1을 함께 고려하는 것이 적절하다. 이에 따라 본 연구에서는 전체 정확도와 클래스별 균형 성능을 종합적으로 고려하여 RandomForest를 최종 모델로 선정하였다.

이 모델의 핵심 결과물은 관광지별 최종 유형과 유형별 확률값이다.  
해당 확률값은 이후 숙소 기반 관광지 추천, 테마 기반 추천, 코스 구성, 사용자 선호 반영 추천 모델의 주요 입력값으로 활용할 수 있다.

In [1]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import joblib

warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 불러오기

In [2]:
DATA_PATH = '../Data/target_tourism_type/남양주시관광지.csv'
OUTPUT_DIR = 'tourism_type_model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(381, 25)


,addr1,addr2,areacode,cat1,cat2,cat3,contentid,contenttypeid,createdtime,firstimage,...,modifiedtime,sigungucode,tel,title,zipcode,lDongRegnCd,lDongSignguCd,lclsSystm1,lclsSystm2,lclsSystm3
0,경기도 남양주시 강변북로632번길 57-28 (수석동),NaN,NaN,NaN,NaN,NaN,798518,39,20090907191546,https://tong.visitkorea.or.kr/cms/resource/99/...,...,20260417133837,NaN,NaN,가든갤러리,12268,41,360.0,FD,FD02,FD020300
1,경기도 남양주시 화도읍 북한강로1570번길 16,NaN,NaN,NaN,NaN,NaN,3045767,28,20231122100649,https://tong.visitkorea.or.kr/cms/resource/53/...,...,20260408150146,NaN,NaN,가람수상스키,12193,41,360.0,LS,LS02,LS021400
2,경기도 남양주시 수동면 비룡로 1603,NaN,31.0,A03,A0302,A03021700,2738718,28,20210901200909,http://tong.visitkorea.or.kr/cms/resource/41/2...,...,20250513155541,9.0,NaN,가족쉼터,12024,41,360.0,AC,AC05,AC050100
3,경기도 남양주시 조안면 다산로747번길 45,NaN,31.0,A05,A0502,A05020100,133832,39,20040220090000,NaN,...,20250404132937,9.0,NaN,감나무집,12283,41,360.0,FD,FD01,FD010100
4,경기도 남양주시 와부읍 경강로 876,NaN,NaN,NaN,NaN,NaN,134729,39,20041201090000,NaN,...,20260317101258,NaN,NaN,개성집,12277,41,360.0,FD,FD01,FD010100


## 2. 기본 전처리

In [3]:
df_model = df.copy()

# 문자열 컬럼 결측 처리
text_cols = ['title', 'addr1', 'addr2', 'firstimage', 'firstimage2', 'tel', 'cpyrhtDivCd',
             'cat1', 'cat2', 'cat3', 'lclsSystm1', 'lclsSystm2', 'lclsSystm3']
for col in text_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna('').astype(str).str.strip()

# 숫자형 변환
num_cols = ['contentid', 'contenttypeid', 'mapx', 'mapy', 'mlevel', 'zipcode',
            'lDongRegnCd', 'lDongSignguCd', 'createdtime', 'modifiedtime']
for col in num_cols:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

# 콘텐츠 유형명 매핑
CONTENT_TYPE_MAP = {
    12: '관광지',
    14: '문화시설',
    15: '축제공연행사',
    25: '여행코스',
    28: '레포츠',
    32: '숙박',
    38: '쇼핑',
    39: '음식점'
}

df_model['content_type_name'] = df_model['contenttypeid'].map(CONTENT_TYPE_MAP).fillna('기타')

# 날짜 파생 변수
for date_col in ['createdtime', 'modifiedtime']:
    df_model[f'{date_col}_dt'] = pd.to_datetime(
        df_model[date_col].astype('Int64').astype(str),
        format='%Y%m%d%H%M%S',
        errors='coerce'
    )

df_model['modified_year'] = df_model['modifiedtime_dt'].dt.year
_df_month = df_model['modifiedtime_dt'].dt.month
df_model['modified_month'] = _df_month

df_model[['title', 'contenttypeid', 'content_type_name', 'modified_year', 'modified_month']].head()

,title,contenttypeid,content_type_name,modified_year,modified_month
0,가든갤러리,39,음식점,2026,4
1,가람수상스키,28,레포츠,2026,4
2,가족쉼터,28,레포츠,2025,5
3,감나무집,39,음식점,2025,4
4,개성집,39,음식점,2026,3


## 3. 관광지 유형 라벨 생성

In [4]:
def make_tourism_type(row):
    content_type = row.get('contenttypeid')
    l1 = str(row.get('lclsSystm1', '')).upper().strip()
    l2 = str(row.get('lclsSystm2', '')).upper().strip()
    l3 = str(row.get('lclsSystm3', '')).upper().strip()
    title = str(row.get('title', '')).lower()

    # 콘텐츠 유형 우선 매핑
    if content_type == 39:
        return '음식'
    if content_type == 38:
        return '쇼핑'
    if content_type == 32:
        return '숙박'
    if content_type == 28:
        return '레저스포츠'
    if content_type == 14:
        return '문화관광'
    if content_type == 15:
        return '축제행사'
    if content_type == 25:
        return '여행코스'

    # 관광지 내부 세부 유형 보정
    code_text = f'{l1} {l2} {l3}'
    if any(k in title for k in ['궁', '릉', '유적', '문화재', '사찰', '절', '묘', '성지', '박물관']):
        return '역사관광'
    if any(k in title for k in ['공원', '수목원', '정원', '숲', '산', '계곡', '호수', '강', '둘레길', '물의정원']):
        return '자연관광'
    if any(k in title for k in ['체험', '캠핑', '농장', '승마', '수상스키', '스키', '레저']):
        return '체험관광'

    # 대분류 코드 보정
    if l1.startswith('VE'):
        return '관광지'
    if l1.startswith('HS'):
        return '숙박'
    if l1.startswith('FD'):
        return '음식'
    if l1.startswith('SH'):
        return '쇼핑'
    if l1.startswith('LS'):
        return '레저스포츠'

    return '관광지'


df_model['target_tourism_type'] = df_model.apply(make_tourism_type, axis=1)

df_model['target_tourism_type'].value_counts()

target_tourism_type
음식       173
쇼핑        82
레저스포츠     43
자연관광      25
문화관광      17
관광지       16
숙박        12
역사관광       8
체험관광       5
Name: count, dtype: int64

## 4. 개선된 정보 완성도 점수 생성

### info_score

`info_score`는 관광지 데이터의 정보 품질을 보강하면 올라감

- `firstimage`, `firstimage2` 이미지 URL 보강
- `tel` 전화번호 보강
- `addr1`, `addr2`, `zipcode` 주소 정보 보강
- `cat1`, `cat2`, `cat3` 또는 `lclsSystm1~3` 분류 코드 보강
- `mapx`, `mapy` 좌표 보강
- 최신 수정일 `modifiedtime` 보강

모델 2에서는 `info_score`가 높은 관광지를 더 신뢰도 높은 추천 후보로 활용할 수 있음

In [5]:
def not_empty(series):
    return series.fillna('').astype(str).str.strip().ne('')

# 정보 보유 여부 컬럼
_df = df_model
_df['has_title'] = not_empty(_df['title']).astype(int)
_df['has_addr1'] = not_empty(_df['addr1']).astype(int)
_df['has_addr2'] = not_empty(_df['addr2']).astype(int)
_df['has_coord'] = (_df['mapx'].notna() & _df['mapy'].notna()).astype(int)
_df['has_contenttype'] = _df['contenttypeid'].notna().astype(int)
_df['has_cat1'] = not_empty(_df['cat1']).astype(int)
_df['has_cat2'] = not_empty(_df['cat2']).astype(int)
_df['has_cat3'] = not_empty(_df['cat3']).astype(int)
_df['has_lcls1'] = not_empty(_df['lclsSystm1']).astype(int)
_df['has_lcls2'] = not_empty(_df['lclsSystm2']).astype(int)
_df['has_lcls3'] = not_empty(_df['lclsSystm3']).astype(int)
_df['has_firstimage'] = not_empty(_df['firstimage']).astype(int)
_df['has_firstimage2'] = not_empty(_df['firstimage2']).astype(int)
_df['has_tel'] = not_empty(_df['tel']).astype(int)
_df['has_zipcode'] = _df['zipcode'].notna().astype(int)
_df['has_modifiedtime'] = _df['modifiedtime_dt'].notna().astype(int)

# 대/중/소분류는 TourAPI cat과 lcls 중 하나라도 있으면 인정
_df['has_main_category'] = ((_df['has_cat1'] == 1) | (_df['has_lcls1'] == 1)).astype(int)
_df['has_mid_category'] = ((_df['has_cat2'] == 1) | (_df['has_lcls2'] == 1)).astype(int)
_df['has_sub_category'] = ((_df['has_cat3'] == 1) | (_df['has_lcls3'] == 1)).astype(int)

# 0~100점 정보 완성도 점수
_df['info_score'] = (
    _df['has_title'] * 10 +
    _df['has_addr1'] * 15 +
    _df['has_addr2'] * 5 +
    _df['has_coord'] * 15 +
    _df['has_contenttype'] * 10 +
    _df['has_main_category'] * 5 +
    _df['has_mid_category'] * 5 +
    _df['has_sub_category'] * 5 +
    _df['has_firstimage'] * 10 +
    _df['has_firstimage2'] * 5 +
    _df['has_tel'] * 5 +
    _df['has_zipcode'] * 5 +
    _df['has_modifiedtime'] * 5
)

# 등급화: 모델 해석 및 시각화용
_df['info_level'] = pd.cut(
    _df['info_score'],
    bins=[-1, 49, 69, 84, 100],
    labels=['낮음', '보통', '높음', '매우 높음']
)

df_model[['title', 'info_score', 'info_level', 'has_firstimage', 'has_tel', 'has_addr1']].head()

,title,info_score,info_level,has_firstimage,has_tel,has_addr1
0,가든갤러리,90,매우 높음,1,0,1
1,가람수상스키,90,매우 높음,1,0,1
2,가족쉼터,90,매우 높음,1,0,1
3,감나무집,75,높음,0,0,1
4,개성집,75,높음,0,0,1


## 5. 모델 입력 변수 구성

In [6]:
TEXT_FEATURES = ['title', 'addr1']
CATEGORICAL_FEATURES = [
    'content_type_name', 'cpyrhtDivCd',
    'cat1', 'cat2', 'cat3',
    'lclsSystm1', 'lclsSystm2', 'lclsSystm3',
    'info_level'
]
NUMERIC_FEATURES = [
    'contenttypeid', 'mapx', 'mapy', 'mlevel', 'zipcode',
    'lDongRegnCd', 'lDongSignguCd',
    'modified_year', 'modified_month',
    'has_firstimage', 'has_firstimage2', 'has_tel', 'has_addr1',
    'has_coord', 'has_main_category', 'has_mid_category', 'has_sub_category',
    'info_score'
]

FEATURES = TEXT_FEATURES + CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = 'target_tourism_type'

# 실제 존재하는 컬럼만 사용
FEATURES = [c for c in FEATURES if c in df_model.columns]
TEXT_FEATURES = [c for c in TEXT_FEATURES if c in FEATURES]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in FEATURES]
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in FEATURES]

X = df_model[FEATURES]
y = df_model[TARGET]

print('입력 변수 수:', len(FEATURES))
print('타깃 클래스:', sorted(y.unique()))

입력 변수 수: 29
타깃 클래스: ['관광지', '레저스포츠', '문화관광', '쇼핑', '숙박', '역사관광', '음식', '자연관광', '체험관광']


## 6. 모델 학습

In [7]:
# 1. Text 피처 제거 (또는 별도로 처리)
# 이 예시에서는 텍스트 피처를 아예 사용하지 않음을 가정합니다.
# 만약 텍스트를 써야 한다면 TfidfVectorizer를 써야 합니다.
TEXT_FEATURES_TO_USE = [] # 비우거나 TF-IDF 파이프라인 구성

# 2. 데이터 분할
class_counts = y.value_counts()
can_stratify = class_counts.min() >= 2

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y if can_stratify else None
)

# 3. 피처 트랜스포머 정의 (Text 제거)
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # sparse_output=False로 설정하여dense array 반환
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[
        # ('text', text_transformer, TEXT_FEATURES_TO_USE), # 텍스트 파이프라인 주석 처리
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
        ('num', numeric_transformer, NUMERIC_FEATURES)
    ],
    remainder='drop'
)

# 4. 모델 정의 및 학습
clf = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced',
        min_samples_leaf=2,
        n_jobs=-1 # 병렬 처리로 학습 속도 향상
    ))
])

clf.fit(X_train, y_train)

# 5. 예측 및 평가
pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
f1_macro = f1_score(y_test, pred, average='macro')
f1_weighted = f1_score(y_test, pred, average='weighted')

print(f'Accuracy: {round(acc, 4)}')
print(f'Macro F1: {round(f1_macro, 4)}')
print(f'Weighted F1: {round(f1_weighted, 4)}')
print(classification_report(y_test, pred))

Accuracy: 0.9481
Macro F1: 0.8413
Weighted F1: 0.9487
              precision    recall  f1-score   support

         관광지       0.50      0.67      0.57         3
       레저스포츠       1.00      1.00      1.00         9
        문화관광       1.00      1.00      1.00         3
          쇼핑       1.00      1.00      1.00        17
          숙박       1.00      1.00      1.00         2
        역사관광       1.00      0.50      0.67         2
          음식       1.00      1.00      1.00        35
        자연관광       0.75      0.60      0.67         5
        체험관광       0.50      1.00      0.67         1

    accuracy                           0.95        77
   macro avg       0.86      0.86      0.84        77
weighted avg       0.96      0.95      0.95        77



## 7. 기존 관광지 전체에 대한 유형별 확률 생성

In [8]:
X_all = df_model[FEATURES]

proba = clf.predict_proba(X_all)
classes = list(clf.named_steps['model'].classes_)

prob_cols = [
    'type_prob_' + str(c).strip().replace('/', '_').replace(' ', '_')
    for c in classes
]

proba_df = pd.DataFrame(proba, columns=prob_cols)

# 1순위 유형과 확률
proba_df['type_top1'] = clf.predict(X_all)
proba_df['type_top1_prob'] = proba.max(axis=1)

# 2순위 유형과 확률
if len(classes) >= 2:
    top2_idx = proba.argsort(axis=1)[:, -2]
    proba_df['type_top2'] = [classes[i] for i in top2_idx]
    proba_df['type_top2_prob'] = [proba[i, top2_idx[i]] for i in range(len(df_model))]
else:
    proba_df['type_top2'] = None
    proba_df['type_top2_prob'] = np.nan

proba_df.head()

,type_prob_관광지,type_prob_레저스포츠,type_prob_문화관광,type_prob_쇼핑,type_prob_숙박,type_prob_역사관광,type_prob_음식,type_prob_자연관광,type_prob_체험관광,type_top1,type_top1_prob,type_top2,type_top2_prob
0,0.029234,0.048552,0.051657,0.033745,0.027832,0.014910,0.773090,0.016819,0.004160,음식,0.773090,문화관광,0.051657
1,0.018758,0.921669,0.024672,0.008095,0.008626,0.006355,0.004009,0.005783,0.002034,레저스포츠,0.921669,문화관광,0.024672
2,0.003838,0.911588,0.009394,0.002752,0.046026,0.004869,0.002019,0.018245,0.001269,레저스포츠,0.911588,숙박,0.046026
3,0.003625,0.003182,0.011415,0.001232,0.002739,0.005527,0.955183,0.009861,0.007237,음식,0.955183,문화관광,0.011415
4,0.022116,0.016336,0.034784,0.006419,0.021244,0.004189,0.871384,0.012185,0.011342,음식,0.871384,문화관광,0.034784


## 8. 모델 1 output 저장

저장 파일은 두 가지입니다.

1. `model1_type_probability_features.csv`  
   - 관광지 식별 정보 + 유형별 확률만 저장

2. `namyangju_dataset_with_type_probabilities.csv`  
   - 기존 관광지 데이터 + 유형별 확률을 모두 포함
   - 모델 2에서 바로 사용 가능


In [9]:
base_cols = [
    'contentid', 'title', 'addr1', 'contenttypeid', 'content_type_name',
    'target_tourism_type', 'mapx', 'mapy',
    'info_score', 'info_level',
    'has_firstimage', 'has_firstimage2', 'has_tel', 'has_addr1',
    'has_coord', 'has_main_category', 'has_mid_category', 'has_sub_category'
]
base_cols = [c for c in base_cols if c in df_model.columns]

model1_output = pd.concat(
    [df_model[base_cols].reset_index(drop=True), proba_df.reset_index(drop=True)],
    axis=1
)

model2_input = pd.concat(
    [df_model.reset_index(drop=True), proba_df.reset_index(drop=True)],
    axis=1
)

model1_output_path = os.path.join(OUTPUT_DIR, 'model1_type_probability_features.csv')
model2_input_path = os.path.join(OUTPUT_DIR, 'namyangju_dataset_with_type_probabilities.csv')
model_path = os.path.join(OUTPUT_DIR, 'tourism_type_classifier.joblib')

model1_output.to_csv(model1_output_path, index=False, encoding='utf-8-sig')
model2_input.to_csv(model2_input_path, index=False, encoding='utf-8-sig')
joblib.dump(clf, model_path)

print('저장 완료')
print(model1_output_path)
print(model2_input_path)
print(model_path)

model1_output.head()

저장 완료
tourism_type_model_outputs/model1_type_probability_features.csv
tourism_type_model_outputs/namyangju_dataset_with_type_probabilities.csv
tourism_type_model_outputs/tourism_type_classifier.joblib


,contentid,title,addr1,contenttypeid,content_type_name,target_tourism_type,mapx,mapy,info_score,info_level,...,type_prob_쇼핑,type_prob_숙박,type_prob_역사관광,type_prob_음식,type_prob_자연관광,type_prob_체험관광,type_top1,type_top1_prob,type_top2,type_top2_prob
0,798518,가든갤러리,경기도 남양주시 강변북로632번길 57-28 (수석동),39,음식점,음식,127.174134,37.587745,90,매우 높음,...,0.033745,0.027832,0.014910,0.773090,0.016819,0.004160,음식,0.773090,문화관광,0.051657
1,3045767,가람수상스키,경기도 남양주시 화도읍 북한강로1570번길 16,28,레포츠,레저스포츠,127.368002,37.647671,90,매우 높음,...,0.008095,0.008626,0.006355,0.004009,0.005783,0.002034,레저스포츠,0.921669,문화관광,0.024672
2,2738718,가족쉼터,경기도 남양주시 수동면 비룡로 1603,28,레포츠,레저스포츠,127.273672,37.755968,90,매우 높음,...,0.002752,0.046026,0.004869,0.002019,0.018245,0.001269,레저스포츠,0.911588,숙박,0.046026
3,133832,감나무집,경기도 남양주시 조안면 다산로747번길 45,39,음식점,음식,127.302971,37.517566,75,높음,...,0.001232,0.002739,0.005527,0.955183,0.009861,0.007237,음식,0.955183,문화관광,0.011415
4,134729,개성집,경기도 남양주시 와부읍 경강로 876,39,음식점,음식,127.234925,37.559514,75,높음,...,0.006419,0.021244,0.004189,0.871384,0.012185,0.011342,음식,0.871384,문화관광,0.034784


## 9. 결과 확인

In [10]:
# 유형별 대표 확률 확인
show_cols = ['title', 'target_tourism_type', 'type_top1', 'type_top1_prob', 'type_top2', 'type_top2_prob', 'info_score']
show_cols += prob_cols[:5]
show_cols = [c for c in show_cols if c in model1_output.columns]

model1_output[show_cols].sort_values('type_top1_prob', ascending=False).head(15)

,title,target_tourism_type,type_top1,type_top1_prob,type_top2,type_top2_prob,info_score,type_prob_관광지,type_prob_레저스포츠,type_prob_문화관광,type_prob_쇼핑,type_prob_숙박
201,쌍둥이해장국,음식,음식,0.988341,문화관광,0.004643,90,0.000974,0.000334,0.004643,0.001734,0.002677
158,브레드쏭 팔당,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
208,아벨,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
334,팔당제빵소,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
218,어나더쥬얼리,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
370,힐링힐스,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
336,팔숲,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
25,나인블럭 뷰 팔당점,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
330,팔당로그,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312
342,포러데이 팔당,음식,음식,0.986518,숙박,0.004312,90,0.001292,0.000334,0.002828,0.001670,0.004312


## 10. Random Forest, CatBoost, XGBoost 모델 비교

In [11]:
# ============================================
# 11. RandomForest / CatBoost / XGBoost 모델 비교
# ============================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# --------------------------------------------
# 1. 데이터 분할
# --------------------------------------------

class_counts = y.value_counts()
can_stratify = class_counts.min() >= 2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if can_stratify else None
)

# --------------------------------------------
# 2. 공통 전처리 파이프라인
# --------------------------------------------

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
        ('num', numeric_transformer, NUMERIC_FEATURES)
    ],
    remainder='drop'
)

# --------------------------------------------
# 3. XGBoost용 Label Encoding
# --------------------------------------------

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# --------------------------------------------
# 4. 비교 모델 정의
# --------------------------------------------

models = {
    'RandomForest': Pipeline([
        ('preprocess', preprocess),
        ('model', RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight='balanced',
            min_samples_leaf=2,
            n_jobs=-1
        ))
    ]),

    'CatBoost': Pipeline([
        ('preprocess', preprocess),
        ('model', CatBoostClassifier(
            iterations=300,
            depth=6,
            learning_rate=0.05,
            loss_function='MultiClass',
            eval_metric='TotalF1',
            random_seed=42,
            verbose=0
        ))
    ]),

    'XGBoost': Pipeline([
        ('preprocess', preprocess),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            objective='multi:softprob',
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        ))
    ])
}

# --------------------------------------------
# 5. 모델 학습 및 평가
# --------------------------------------------

results = []
trained_models = {}

for model_name, model in models.items():
    print(f'\n==============================')
    print(f'{model_name} 학습 시작')
    print(f'==============================')

    if model_name == 'XGBoost':
        model.fit(X_train, y_train_encoded)
        pred_encoded = model.predict(X_test)
        pred = label_encoder.inverse_transform(pred_encoded.astype(int))
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        # CatBoost 예측 결과가 2차원으로 나오는 경우 처리
        if len(np.array(pred).shape) > 1:
            pred = np.array(pred).ravel()

    acc = accuracy_score(y_test, pred)
    f1_macro = f1_score(y_test, pred, average='macro')
    f1_weighted = f1_score(y_test, pred, average='weighted')

    results.append({
        'model': model_name,
        'accuracy': acc,
        'macro_f1': f1_macro,
        'weighted_f1': f1_weighted
    })

    trained_models[model_name] = model

    print(f'Accuracy    : {acc:.4f}')
    print(f'Macro F1    : {f1_macro:.4f}')
    print(f'Weighted F1 : {f1_weighted:.4f}')
    print('\n분류 리포트')
    print(classification_report(y_test, pred))

# --------------------------------------------
# 6. 비교 결과표
# --------------------------------------------

result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by='macro_f1', ascending=False)

result_df


RandomForest 학습 시작
Accuracy    : 0.9481
Macro F1    : 0.8413
Weighted F1 : 0.9487

분류 리포트
              precision    recall  f1-score   support

         관광지       0.50      0.67      0.57         3
       레저스포츠       1.00      1.00      1.00         9
        문화관광       1.00      1.00      1.00         3
          쇼핑       1.00      1.00      1.00        17
          숙박       1.00      1.00      1.00         2
        역사관광       1.00      0.50      0.67         2
          음식       1.00      1.00      1.00        35
        자연관광       0.75      0.60      0.67         5
        체험관광       0.50      1.00      0.67         1

    accuracy                           0.95        77
   macro avg       0.86      0.86      0.84        77
weighted avg       0.96      0.95      0.95        77


CatBoost 학습 시작
Accuracy    : 0.9221
Macro F1    : 0.7848
Weighted F1 : 0.9231

분류 리포트
              precision    recall  f1-score   support

         관광지       0.25      0.33      0.29         3
       레

,model,accuracy,macro_f1,weighted_f1
0,RandomForest,0.948052,0.841270,0.948670
2,XGBoost,0.948052,0.828956,0.945927
1,CatBoost,0.922078,0.784832,0.923109
